# CXR-Recall D9: does better retrieval improve classification?

**Background.** Over 3 seeds, CXR-Recall (fine-tuned ViT encoder, frozen; retrieval with that same encoder) did not beat the same encoder without retrieval: −0.0042 ± 0.0060 test macro AUROC. The encoder comparison then showed RAD-DINO retrieves abnormal cases ~40% better (P@5 0.273 vs 0.194). This notebook asks whether that better retrieval turns into better diagnosis.

**Design: same as D3, with RAD-DINO as the frozen shared encoder.** Every arm sees the same frozen RAD-DINO [CLS] embedding of the query, so the only thing that differs is the evidence pathway:

| Arm | Query features | Evidence |
|---|---|---|
| `probe` (baseline) | RAD-DINO | none: linear head |
| `retrieval` (CXR-Recall) | RAD-DINO | top-5 RAD-DINO neighbours from train (image + label tokens, joint attention, residual fusion) |
| `random_control` | RAD-DINO | 5 **random** train cases, same architecture as `retrieval` |

- `retrieval − probe`: does the full design help?
- `retrieval − random_control`: is any gain from **retrieval**, rather than from the extra parameters of the evidence encoder? This control is new; the earlier ViT runs didn't have it.

**Fairness and leakage.** Train queries never see their own study as a neighbour (`LEAKAGE CHECK` printed). Val/test studies are never in the gallery. Each arm gets the same learning-rate sweep (selected on val per seed), up to 200 epochs with patience 15, and a warning if any selected run hits the cap. 5 training seeds; per-seed test margins are paired (same 743 test images).

**Settings:** GPU on, **Internet on** (RAD-DINO download). Attach "Chest X-rays Indiana University" (`raddar/chest-xrays-indiana-university`). Use **Save Version, Save & Run All**. Runtime about 25-40 min on a T4: RAD-DINO extraction ~10 min, then training on cached embeddings.

**What to paste back:** the final `RESULT_JSON:` line (or the aggregate table above it).

In [ ]:
import os, glob, json, time, copy, random, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from scipy import stats

print('torch:', torch.__version__, ' cuda available:', torch.cuda.is_available())

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

## 1. Configuration

In [ ]:
INPUT_DIR = '/kaggle/input'
WORK_DIR = '/kaggle/working'
RADDINO_ID = 'microsoft/rad-dino'   # pretrained on MIMIC-CXR, CheXpert, NIH, PadChest, BRAX -- not Open-I
EXTRACT_BATCH = 32
K = 5
SEEDS = [0, 1, 2, 3, 4]
LRS = [1e-4, 3e-4, 1e-3]            # swept per arm and seed, chosen on val
MAX_EPOCHS, PATIENCE = 200, 15
BATCH_SIZE = 64
ARMS = ['probe', 'retrieval', 'random_control']
# ViT-based system, 3 seeds (ARCHITECTURE.md section 4), for context only -- not paired with this run
VIT_REFERENCE = {'baseline_test': 0.7131, 'cxr_recall_test': 0.7090, 'margin_test': -0.0042}
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 2. Locate the attached dataset (identical to the multi-seed notebook)

In [ ]:
import os

candidates = os.listdir(INPUT_DIR)
assert candidates, "No dataset attached — use '+ Add Data' and search for the Indiana University chest x-ray collection first."
print('attached datasets:', candidates)

DATASET_DIR = os.path.join(INPUT_DIR, candidates[0])
print('using:', DATASET_DIR)
for root, dirs, files in os.walk(DATASET_DIR):
    print(root, '->', dirs, files[:5])

In [ ]:
import glob

csv_files = glob.glob(os.path.join(DATASET_DIR, '**', '*.csv'), recursive=True)
png_dirs = sorted(set(os.path.dirname(p) for p in glob.glob(os.path.join(DATASET_DIR, '**', '*.png'), recursive=True)))
print('csv files:', csv_files)
print('folders containing pngs:', png_dirs)

## 3. Manifest, label vocabulary, fixed study-level split (identical to the multi-seed notebook)

In [ ]:
reports_csv = [f for f in csv_files if 'report' in os.path.basename(f).lower()]
projections_csv = [f for f in csv_files if 'projection' in os.path.basename(f).lower()]
assert reports_csv and projections_csv, 'Could not auto-find reports/projections CSVs -- inspect csv_files above and set paths manually.'

reports_df = pd.read_csv(reports_csv[0])
proj_df = pd.read_csv(projections_csv[0])

UID_COL, FINDINGS_COL, IMPRESSION_COL, TAGS_COL = 'uid', 'findings', 'impression', 'Problems'
PROJ_UID_COL, FILENAME_COL = 'uid', 'filename'

merged = proj_df.merge(reports_df, left_on=PROJ_UID_COL, right_on=UID_COL, suffixes=('', '_report'))

def build_report_text(row):
    parts = [str(row.get(FINDINGS_COL, '')), str(row.get(IMPRESSION_COL, ''))]
    return ' '.join(p for p in parts if p and p.lower() != 'nan').strip()

def build_tags(row):
    raw = row.get(TAGS_COL, '')
    if pd.isna(raw):
        return ''
    sep = ';' if ';' in str(raw) else ','
    return ';'.join(t.strip() for t in str(raw).split(sep) if t.strip())

manifest = pd.DataFrame({
    'image_path': merged[FILENAME_COL],
    'patient_id': merged[UID_COL].astype(str),
    'report_text': merged.apply(build_report_text, axis=1),
    'mesh_tags': merged.apply(build_tags, axis=1),
})

IMAGE_ROOT = png_dirs[0]
print('manifest rows:', len(manifest), ' image root:', IMAGE_ROOT)
manifest.head()

In [ ]:
from collections import Counter
import numpy as np

def build_mesh_vocab(tag_lists, top_k=14):
    counts = Counter(tag for tags in tag_lists for tag in tags)
    return [tag for tag, _ in counts.most_common(top_k)]

def mesh_tags_to_vector(tags, vocab):
    tag_set = set(tags)
    return np.array([1.0 if v in tag_set else 0.0 for v in vocab], dtype=np.float32)

In [ ]:
def patient_level_split(df, patient_id_col, ratios=(0.8, 0.1, 0.1), seed=42):
    patients = sorted(df[patient_id_col].unique())
    rng = np.random.default_rng(seed)
    rng.shuffle(patients)
    n = len(patients)
    n_train, n_val = int(n * ratios[0]), int(n * ratios[1])
    train_p = set(patients[:n_train])
    val_p = set(patients[n_train:n_train + n_val])
    test_p = set(patients[n_train + n_val:])
    return (df[df[patient_id_col].isin(train_p)],
            df[df[patient_id_col].isin(val_p)],
            df[df[patient_id_col].isin(test_p)])

train_df, val_df, test_df = patient_level_split(manifest, 'patient_id')
print('split sizes:', len(train_df), len(val_df), len(test_df))

train_tags = [t.split(';') if t else [] for t in train_df['mesh_tags']]
label_vocab = build_mesh_vocab(train_tags, top_k=14)
print('label vocab:', label_vocab)

EXPECTED_VOCAB = ['normal', 'Lung', 'Opacity', 'Cardiomegaly', 'Pulmonary Atelectasis', 'Calcinosis',
                  'Calcified Granuloma', 'Thoracic Vertebrae', 'Cicatrix', 'Spine', 'Pleural Effusion',
                  'Aorta', 'Markings', 'Diaphragm']
if INPUT_DIR == '/kaggle/input':
    assert (len(train_df), len(val_df), len(test_df)) == (5977, 746, 743), 'split differs from every other Phase 1 run'
    assert label_vocab == EXPECTED_VOCAB, 'label vocabulary differs from every other Phase 1 run'
    print('split and vocabulary match the documented Phase 1 runs')

## 4. RAD-DINO [CLS] embeddings (frozen, extracted once)

In [ ]:
def tags_to_vec(tag_str):
    tags = tag_str.split(';') if tag_str else []
    return mesh_tags_to_vector(tags, label_vocab)

class PathLabelDataset(Dataset):
    def __init__(self, df, preprocess):
        self.df = df.reset_index(drop=True)
        self.preprocess = preprocess
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        return self.preprocess(os.path.join(IMAGE_ROOT, row['image_path'])), tags_to_vec(row['mesh_tags'])

def make_raddino():
    from transformers import AutoImageProcessor, AutoModel
    processor = AutoImageProcessor.from_pretrained(RADDINO_ID)
    model = AutoModel.from_pretrained(RADDINO_ID).to(device).eval()
    def preprocess(path):
        return processor(images=Image.open(path).convert('RGB'), return_tensors='pt')['pixel_values'][0]
    return model, preprocess

@torch.no_grad()
def extract(model, preprocess, df):
    loader = DataLoader(PathLabelDataset(df, preprocess), batch_size=EXTRACT_BATCH, shuffle=False, num_workers=2)
    feats, labels = [], []
    for x, y in loader:
        feats.append(model(pixel_values=x.to(device)).pooler_output.float().cpu().numpy())
        labels.append(y.numpy())
    return np.concatenate(feats), np.concatenate(labels)

split_dfs = {'train': train_df, 'val': val_df, 'test': test_df}
EMB_PATH = os.path.join(WORK_DIR, 'raddino_cls_embeddings.npz')
t0 = time.time()
raddino, preprocess = make_raddino()
EMB, LBL = {}, {}
for s, df in split_dfs.items():
    EMB[s], LBL[s] = extract(raddino, preprocess, df)
del raddino; gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
PIDS = {s: df['patient_id'].astype(str).values for s, df in split_dfs.items()}
np.savez_compressed(EMB_PATH, **{f'{s}_emb': EMB[s] for s in EMB}, **{f'{s}_lbl': LBL[s] for s in LBL},
                    **{f'{s}_pid': PIDS[s] for s in PIDS})
print({s: EMB[s].shape for s in EMB}, f'extracted in {(time.time() - t0) / 60:.1f} min, saved to {EMB_PATH}')

## 5. Neighbours

`retrieval`: cosine top-5 among train images. Train queries skip every image from their own study. `random_control`: 5 random train images per query (also never from the query's own study), redrawn per seed and fixed for that seed's training.

In [ ]:
def l2n(x):
    return x / np.linalg.norm(x, axis=1, keepdims=True)

TRAIN_N = torch.from_numpy(l2n(EMB['train'])).to(device)

def topk_train(q, k):
    out = []
    for i in range(0, len(q), 1024):
        sims = torch.from_numpy(l2n(q[i:i + 1024])).to(device) @ TRAIN_N.T
        out.append(torch.topk(sims, k, dim=1).indices.cpu().numpy())
    return np.concatenate(out)

train_pids = PIDS['train']
MAX_VIEWS = int(train_df.groupby('patient_id').size().max())
raw = topk_train(EMB['train'], K + MAX_VIEWS)
nb_train, same_study_hits = [], 0
for i, row in enumerate(raw):
    index_only = [j for j in row if j != i][:K]
    same_study_hits += int(any(train_pids[j] == train_pids[i] for j in index_only))
    keep = [j for j in row if train_pids[j] != train_pids[i]][:K]
    assert len(keep) == K, f'query {i}: fewer than {K} neighbours outside its study'
    nb_train.append(keep)
NB_RETR = {'train': np.array(nb_train), 'val': topk_train(EMB['val'], K), 'test': topk_train(EMB['test'], K)}
leakage_rate = same_study_hits / len(train_pids)
print(f'LEAKAGE CHECK: {same_study_hits}/{len(train_pids)} ({leakage_rate:.1%}) train queries would have retrieved '
      f'their own study without the exclusion (excluded below)')
assert not any(train_pids[j] == train_pids[i] for i, row in enumerate(NB_RETR['train']) for j in row)

def random_neighbours(seed):
    rng = np.random.default_rng(1000 + seed)
    out = {}
    for s in split_dfs:
        q_pids = PIDS[s]
        nb = np.empty((len(q_pids), K), dtype=int)
        for i in range(len(q_pids)):
            while True:
                c = rng.choice(len(train_pids), K, replace=False)
                if not np.any(train_pids[c] == q_pids[i]):
                    break
            nb[i] = c
        out[s] = nb
    return out

def p_at_k(nb, q_lbl):
    return float(((LBL['train'][nb] * q_lbl[:, None, :]).sum(-1) > 0).mean())

_rnd = random_neighbours(0)
for s in ('val', 'test'):
    print(f'{s}: retrieval P@{K} = {p_at_k(NB_RETR[s], LBL[s]):.4f}   random P@{K} = {p_at_k(_rnd[s], LBL[s]):.4f}')
print('(encoder comparison, raddino_cls: val 0.3252, test 0.3211)')

## 6. Models (EvidenceEncoder copied verbatim from `src/models/evidence_encoder.py`)

In [ ]:
"""Evidence encoder + cross-attention fusion for Arm 3 (CXR-Recall). Takes
the query's own image embedding plus information about its k retrieved
neighbors (from Arm 2's FAISS index) and produces a fused representation
that the classification head consumes -- this is what makes retrieval
causally relevant to the diagnosis, unlike Arm 2's display-only retrieval.

Three evidence modes, in order of what was actually tried (see RESULTS.md):
  - label-only (Arm 3): each neighbor is represented purely by its 14-dim
    multi-hot label vector. Causally used but net *hurt* accuracy -- ~70% of
    retrieved neighbors don't share a label with the query, and label-only
    evidence gives the model no way to tell a strong visual match from a
    weak one before trusting it.
  - early fusion, label+image (Arm 3b): each neighbor's image embedding
    (already computed during retrieval) and label vector are concatenated
    and projected into ONE combined vector per neighbor *before* attention
    runs. Closed ~82% of Arm 3's gap to Arm 1. The image embedding is passed
    through its own learned `image_proj` rather than used raw, as a
    mitigation for the "shared representation" collapse risk noted in the
    KNN literature (sharing retrieval and classification representations
    directly can hurt performance).
  - joint attention, label+image (Arm 3c): image and label information for
    each neighbor are kept as SEPARATE attendable tokens (2k tokens total,
    not k pre-fused ones), tagged with a learned modality embedding (so the
    model knows which token is an image vs. a label) and a learned
    neighbor-position embedding (so the model knows which image/label pair
    belongs to the same neighbor). This lets the query attend to a specific
    neighbor's image independently of that neighbor's label, rather than
    having the two pre-blended by a fixed linear combination before
    attention ever runs -- a strictly more expressive mechanism than early
    fusion, at the cost of more attendable items (2k vs k) on a small
    dataset, which is worth watching for overfitting."""

import torch
import torch.nn as nn


class EvidenceEncoder(nn.Module):
    def __init__(self, num_labels: int, embed_dim: int, num_heads: int = 4,
                 use_image_embeddings: bool = False, image_embed_dim: int = None,
                 joint_attention: bool = False, max_k: int = 16):
        super().__init__()
        self.use_image_embeddings = use_image_embeddings
        self.joint_attention = joint_attention
        self.label_proj = nn.Linear(num_labels, embed_dim)

        if use_image_embeddings:
            image_embed_dim = image_embed_dim or embed_dim
            self.image_proj = nn.Linear(image_embed_dim, embed_dim)
            if joint_attention:
                self.modality_embed = nn.Embedding(2, embed_dim)  # 0 = image, 1 = label
                self.neighbor_pos_embed = nn.Embedding(max_k, embed_dim)
            else:
                self.combine = nn.Linear(embed_dim * 2, embed_dim)
        elif joint_attention:
            raise ValueError("joint_attention=True requires use_image_embeddings=True")

        self.attention = nn.MultiheadAttention(embed_dim, num_heads=num_heads, batch_first=True)

    def forward(self, query_embedding: torch.Tensor, evidence_labels: torch.Tensor,
                evidence_image_embeddings: torch.Tensor = None) -> torch.Tensor:
        """query_embedding: (batch, embed_dim). evidence_labels: (batch, k, num_labels).
        evidence_image_embeddings: (batch, k, image_embed_dim), required iff
        use_image_embeddings=True -- the retrieved neighbors' own image
        embeddings, precomputed outside this module during retrieval."""
        label_embed = self.label_proj(evidence_labels)  # (batch, k, embed_dim)

        if self.use_image_embeddings:
            if evidence_image_embeddings is None:
                raise ValueError("use_image_embeddings=True requires evidence_image_embeddings")
            image_embed = self.image_proj(evidence_image_embeddings)  # (batch, k, embed_dim)

            if self.joint_attention:
                k = evidence_labels.shape[1]
                device = evidence_labels.device
                pos = self.neighbor_pos_embed(torch.arange(k, device=device)).unsqueeze(0)  # (1, k, embed_dim)
                image_type = self.modality_embed.weight[0]  # (embed_dim,)
                label_type = self.modality_embed.weight[1]  # (embed_dim,)
                image_tok = image_embed + pos + image_type
                label_tok = label_embed + pos + label_type
                evidence_embed = torch.cat([image_tok, label_tok], dim=1)  # (batch, 2k, embed_dim) -- separately attendable
            else:
                evidence_embed = self.combine(torch.cat([image_embed, label_embed], dim=-1))  # (batch, k, embed_dim)
        else:
            evidence_embed = label_embed

        query = query_embedding.unsqueeze(1)  # (batch, 1, embed_dim)
        context, _weights = self.attention(query, evidence_embed, evidence_embed)
        return context.squeeze(1)  # (batch, embed_dim)

In [ ]:
D = EMB['train'].shape[1]
MU, SD = EMB['train'].mean(0), EMB['train'].std(0) + 1e-6
X = {s: torch.from_numpy((EMB[s] - MU) / SD).float().to(device) for s in EMB}   # standardized with train stats
Y = {s: torch.from_numpy(LBL[s]).float().to(device) for s in LBL}
L = len(label_vocab)

class Probe(nn.Module):
    """No retrieval: linear head on the frozen query embedding (as Arm 1's head)."""
    def __init__(self):
        super().__init__()
        self.head = nn.Linear(D, L)
    def forward(self, q, ev_lbl=None, ev_img=None):
        return self.head(q)

class Fusion(nn.Module):
    """Arm 3c-frozen on cached embeddings: joint attention over image/label tokens, residual fusion, linear head."""
    def __init__(self):
        super().__init__()
        self.evidence_encoder = EvidenceEncoder(L, D, use_image_embeddings=True, image_embed_dim=D,
                                                joint_attention=True, max_k=K)
        self.head = nn.Linear(D, L)
    def forward(self, q, ev_lbl, ev_img):
        return self.head(q + self.evidence_encoder(q, ev_lbl, ev_img))

def macro_auroc(y_true, y_score):
    scores = [roc_auc_score(y_true[:, i], y_score[:, i]) for i in range(y_true.shape[1])
              if len(np.unique(y_true[:, i])) == 2]
    return float(np.mean(scores)) if scores else float('nan')

def logits(model, split, idx, nb, zero=None):
    q = X[split][idx]
    if isinstance(model, Probe):
        return model(q)
    n = nb[split][idx]
    ev_img, ev_lbl = X['train'][n], Y['train'][n]
    if zero in ('all', 'img'):
        ev_img = torch.zeros_like(ev_img)
    if zero == 'all':
        ev_lbl = torch.zeros_like(ev_lbl)
    return model(q, ev_lbl, ev_img)

@torch.no_grad()
def probs(model, split, nb, zero=None):
    model.eval()
    idx = torch.arange(len(Y[split]), device=device)
    return torch.sigmoid(logits(model, split, idx, nb, zero)).cpu().numpy()

def train_one(arm, seed, lr, nb):
    set_seed(seed)
    model = (Probe() if arm == 'probe' else Fusion()).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.BCEWithLogitsLoss()
    gen = torch.Generator().manual_seed(seed)
    n = len(Y['train'])
    best, best_state, best_epoch, bad, epoch = -1.0, None, 0, 0, 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        perm = torch.randperm(n, generator=gen).to(device)
        for b in range(0, n, BATCH_SIZE):
            idx = perm[b:b + BATCH_SIZE]
            opt.zero_grad()
            loss = crit(logits(model, 'train', idx, nb), Y['train'][idx])
            loss.backward()
            opt.step()
        val = macro_auroc(LBL['val'], probs(model, 'val', nb))
        if val > best:
            best, best_state, best_epoch, bad = val, copy.deepcopy(model.state_dict()), epoch, 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    hit_cap = bad < PATIENCE   # loop ran out of epochs before early stopping triggered
    model.load_state_dict(best_state)
    return model, {'val': best, 'epoch': best_epoch, 'stopped': epoch, 'hit_cap': hit_cap, 'lr': lr}

print('models defined; embedding dim', D, ' labels', L)

## 7. Train every arm, every seed

In [ ]:
results = []
t_all = time.time()
for seed in SEEDS:
    t0 = time.time()
    nbs = {'probe': NB_RETR, 'retrieval': NB_RETR, 'random_control': random_neighbours(seed)}
    nbs = {a: {s: torch.as_tensor(v, device=device) for s, v in d.items()} for a, d in nbs.items()}  # index on the GPU
    r = {'seed': seed}
    for arm in ARMS:
        best_model, best_info = None, None
        for lr in LRS:
            model, info = train_one(arm, seed, lr, nbs[arm])
            print(f'[seed {seed} | {arm:14s}] lr={lr:.0e}  best val={info["val"]:.4f} (epoch {info["epoch"]}, '
                  f'stopped {info["stopped"]}{", HIT CAP" if info["hit_cap"] else ""})')
            if best_info is None or info['val'] > best_info['val']:
                best_model, best_info = model, info
        test = macro_auroc(LBL['test'], probs(best_model, 'test', nbs[arm]))
        r.update({f'{arm}_val': round(best_info['val'], 4), f'{arm}_test': round(test, 4),
                  f'{arm}_lr': best_info['lr'], f'{arm}_epoch': best_info['epoch'], f'{arm}_hit_cap': best_info['hit_cap']})
        if arm != 'probe':   # evidence-ablation shift on val, same measurement as every prior run
            real = probs(best_model, 'val', nbs[arm])
            r[f'{arm}_shift_full'] = round(float(np.abs(real - probs(best_model, 'val', nbs[arm], 'all')).mean()), 4)
            r[f'{arm}_shift_img'] = round(float(np.abs(real - probs(best_model, 'val', nbs[arm], 'img')).mean()), 4)
        print(f'[seed {seed} | {arm:14s}] chosen lr={best_info["lr"]:.0e}  val={best_info["val"]:.4f}  TEST={test:.4f}')
    r['margin_retr_vs_probe'] = round(r['retrieval_test'] - r['probe_test'], 4)
    r['margin_retr_vs_random'] = round(r['retrieval_test'] - r['random_control_test'], 4)
    r['margin_random_vs_probe'] = round(r['random_control_test'] - r['probe_test'], 4)
    r['minutes'] = round((time.time() - t0) / 60, 1)
    results.append(r)
    print(f'RESULT_SEED_JSON: {json.dumps(r)}\n')
print(f'all seeds done in {(time.time() - t_all) / 60:.1f} min')

## 8. Aggregate

In [ ]:
n = len(results)
print(f'=== D9: RAD-DINO frozen encoder, {n} seeds, TEST macro AUROC ===')
print('seed |  probe   retrieval  random_ctrl | retr-probe  retr-random  random-probe | img share (retr)')
for r in results:
    share = r['retrieval_shift_img'] / r['retrieval_shift_full'] if r['retrieval_shift_full'] else float('nan')
    print(f"{r['seed']:>4} | {r['probe_test']:.4f}   {r['retrieval_test']:.4f}    {r['random_control_test']:.4f}    | "
          f"{r['margin_retr_vs_probe']:+.4f}     {r['margin_retr_vs_random']:+.4f}      {r['margin_random_vs_probe']:+.4f}     | {share:.1%}")

def agg(xs):
    xs = np.array(xs, dtype=float)
    out = {'mean': round(float(xs.mean()), 4), 'std': round(float(xs.std(ddof=1)), 4) if len(xs) > 1 else None}
    return out

def margin_stats(key):
    m = np.array([r[key] for r in results], dtype=float)
    out = {**agg(m), 'positive': int((m > 0).sum()), 'n': len(m)}
    if len(m) >= 2 and m.std(ddof=1) > 0:
        half = stats.t.ppf(0.975, len(m) - 1) * m.std(ddof=1) / np.sqrt(len(m))
        out.update({'ci95': [round(float(m.mean() - half), 4), round(float(m.mean() + half), 4)],
                    'p_ttest': round(float(stats.ttest_1samp(m, 0.0).pvalue), 4)})
    return out

summary = {
    'arms_test': {arm: agg([r[f'{arm}_test'] for r in results]) for arm in ARMS},
    'arms_val': {arm: agg([r[f'{arm}_val'] for r in results]) for arm in ARMS},
    'margins_test': {k: margin_stats(k) for k in ('margin_retr_vs_probe', 'margin_retr_vs_random', 'margin_random_vs_probe')},
    'retrieval_p_at_5': {s: round(p_at_k(NB_RETR[s], LBL[s]), 4) for s in ('val', 'test')},
    'image_share_retrieval': agg([r['retrieval_shift_img'] / r['retrieval_shift_full'] for r in results if r['retrieval_shift_full']]),
    'leakage_rate': round(leakage_rate, 4),
    'any_selected_run_hit_cap': any(r[f'{a}_hit_cap'] for r in results for a in ARMS),
    'vit_reference_not_paired': VIT_REFERENCE,
    'per_seed': results,
}
print()
for arm in ARMS:
    print(f"{arm:15s} TEST {summary['arms_test'][arm]['mean']:.4f} +- {summary['arms_test'][arm]['std']}")
for k, v in summary['margins_test'].items():
    print(f"{k:24s} {v['mean']:+.4f} +- {v['std']}  CI {v.get('ci95')}  p={v.get('p_ttest')}  positive {v['positive']}/{v['n']}")
if summary['any_selected_run_hit_cap']:
    print('\nWARNING: a selected run hit the epoch cap -- it may be under-trained; raise MAX_EPOCHS before interpreting.')

rp, rr = summary['margins_test']['margin_retr_vs_probe'], summary['margins_test']['margin_retr_vs_random']
def excludes_zero_above(v):
    return 'ci95' in v and v['ci95'][0] > 0
if excludes_zero_above(rp) and excludes_zero_above(rr):
    verdict = 'Better retrieval helps: retrieval beats both the no-retrieval baseline and the random-neighbour control.'
elif excludes_zero_above(rp):
    verdict = 'Retrieval beats the baseline, but not clearly the random-neighbour control: the gain may come from the extra parameters, not from retrieval.'
else:
    verdict = 'No reliable gain from retrieval even with the better encoder.'
summary['verdict'] = verdict
print('\nVERDICT:', verdict)

with open(os.path.join(WORK_DIR, 'd9_results.json'), 'w') as f:
    json.dump(summary, f, indent=2, default=float)
print('RESULT_JSON:', json.dumps(summary, default=float))

## How to read this

- **Main number: `margin_retr_vs_probe`.** Positive with a CI above zero means the retrieval-augmented design beats the same RAD-DINO features used alone.
- **The control, `margin_retr_vs_random`.** If retrieval beats the probe but not the random-neighbour control, the gain is from the evidence encoder's extra parameters (or regularisation), not from finding similar cases.
- **Leakage and convergence:** `LEAKAGE CHECK` should be ~1%. `any_selected_run_hit_cap` must be false.
- **The ViT numbers are context only.** They come from a different run, so compare RAD-DINO and ViT only loosely.